# Import Libraries

In [ ]:
from pathlib import Path

import pandas as pd
from tqdm import tqdm

In [ ]:
BASE_DIR = Path(".")
SCREENSHOTS_DIR = BASE_DIR / "screenshots"
CSV_PATH = BASE_DIR / "dataset" / "dataset_crawl_070526_with_images_path.csv"

SIZE_THRESHOLD = 100_000  # 100 KB

# Scan & Remove Small PNGs

PNGs < 100 KB dihapus, diganti file `.error`.

In [ ]:
removed = []
kept = 0

for tag in ["gambling", "non-gambling"]:
    img_dir = SCREENSHOTS_DIR / tag
    if not img_dir.is_dir():
        continue

    png_files = sorted(img_dir.glob("*.png"))
    for fpath in tqdm(png_files, desc=f"Scanning {tag}"):
        sz = fpath.stat().st_size
        if sz < SIZE_THRESHOLD:
            fpath.unlink()
            error_path = fpath.with_suffix(".error")
            error_path.touch()
            rel = f"model/screenshots/{tag}/{fpath.name}"
            removed.append(rel)
        else:
            kept += 1

print(f"\nPNG removed (< {SIZE_THRESHOLD / 1000}KB): {len(removed)}")
print(f"PNG kept   (\u2265 {SIZE_THRESHOLD / 1000}KB): {kept}")

# Update CSV

In [ ]:
if not removed:
    print("Nothing to clean. Exiting.")
else:
    df = pd.read_csv(CSV_PATH)
    mask = df["Screenshot_Path"].isin(removed)
    updated_rows = mask.sum()

    df.loc[mask, "Screenshot_Path"] = ""
    df.loc[mask, "Status"] = "error"
    df.to_csv(CSV_PATH, index=False)

    print(f"CSV rows updated: {updated_rows}")
    print(f"Total CSV rows: {len(df)}")
    print(f"Rows with Screenshot_Path after cleaning: {(df['Screenshot_Path'].notna() & (df['Screenshot_Path'] != '')).sum()}")